# 07 迁移学习实战

## 什么是迁移学习？

**迁移学习（Transfer Learning）** 是深度学习中最实用的技术之一。它的核心思想是：将在大规模数据集（如ImageNet，1400万张图片）上预训练好的模型的知识，迁移到我们的目标任务上。

### 为什么需要迁移学习？

1. **数据不足**：实际项目中，我们通常只有几百到几千张标注图片，远不足以从头训练一个深度网络
2. **计算资源有限**：在ImageNet上训练ResNet-50需要多块GPU运行数天
3. **预训练特征通用性强**：CNN浅层学到的边缘、纹理等特征对几乎所有视觉任务都有用

### 迁移学习的两种策略

| 策略 | 做法 | 优点 | 适用场景 |
|------|------|------|----------|
| **特征提取** | 冻结卷积层，只训练分类头 | 快速、不易过拟合 | 小数据集，与预训练数据相似 |
| **微调** | 解冻部分卷积层，用更小学习率训练 | 性能更好 | 中等数据集，与预训练数据有差异 |

本教程将带你实践完整的迁移学习流程：加载预训练模型 → 特征提取训练 → 微调优化 → Grad-CAM可视化。

In [ ]:
# ===== 环境准备 =====
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import DataLoader
import torchvision
import torchvision.models as models
import torchvision.transforms as T
from torchvision import datasets, transforms

# Albumentations 数据增强库
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    ALBU_AVAILABLE = True
except ImportError:
    ALBU_AVAILABLE = False
    print("警告: albumentations 未安装，将使用 torchvision transforms")

import cv2
import copy
import time
from collections import defaultdict
from pathlib import Path

# 字体设置
mpl.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
mpl.rcParams['axes.unicode_minus'] = False

# 设备
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  |  torchvision {torchvision.__version__}")
print(f"计算设备: {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
if not ALBU_AVAILABLE:
    print("  pip install albumentations  # 安装数据增强库以获得最佳效果")


## 加载预训练的ResNet18

ResNet（残差网络）是2015年ImageNet比赛的冠军架构，其核心创新是**残差连接（Skip Connection）**，解决了深层网络的梯度消失问题。

ResNet18是最轻量的ResNet变体，只有11.7M参数，适合快速实验和CPU训练。

In [ ]:
# ===== 加载预训练ResNet18 =====

# 使用最新的weights API加载ImageNet预训练权重
model_resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

print("=" * 60)
print("ResNet18 模型架构")
print("=" * 60)
print(model_resnet18)

# 统计参数
total_params = sum(p.numel() for p in model_resnet18.parameters())
trainable_params = sum(p.numel() for p in model_resnet18.parameters() if p.requires_grad)
print(f"\n总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

# 查看各层
print("\nResNet18 主要组件:")
for name, module in model_resnet18.named_children():
    params = sum(p.numel() for p in module.parameters())
    print(f"  {name:<15} {params:>10,} 参数")

# ImageNet的1000个类别
print(f"\n原始分类器输出: 1000类 (ImageNet)")
print(f"最后一层: {model_resnet18.fc}")


## 冻结特征层并替换分类器

迁移学习的核心操作：冻结预训练的卷积层参数，只训练新替换的分类头。这样卷积层提取的特征保持不变，只学习如何将这些特征映射到我们的目标类别。

对于ResNet18，最后一层是 `fc`（全连接层），输出1000维（ImageNet的1000类）。我们需要替换为适合自己任务的分类器。

In [ ]:
# ===== 冻结特征层，替换分类器 =====

def build_transfer_model(num_classes, model_name='resnet18', freeze_backbone=True):
    '''
    构建迁移学习模型。

    参数:
        num_classes: 目标分类类别数
        model_name: 'resnet18' | 'resnet50' | 'efficientnet_b0'
        freeze_backbone: True=特征提取模式, False=微调模式

    返回:
        model: 修改后的模型
    '''
    if model_name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features

        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False

        # 替换分类头
        model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    elif model_name == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        in_features = model.fc.in_features

        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False

        model.fc = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    elif model_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        in_features = model.classifier[1].in_features

        if freeze_backbone:
            for param in model.parameters():
                param.requires_grad = False

        model.classifier = nn.Sequential(
            nn.Dropout(p=0.3, inplace=True),
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    else:
        raise ValueError(f"不支持的模型: {model_name}")

    return model


# ---- 演示 ----
NUM_CLASSES = 5  # 假设5个宠物品种
CLASS_NAMES = ['暹罗猫', '波斯猫', '金毛犬', '哈士奇', '柯基犬']

model_fe = build_transfer_model(NUM_CLASSES, model_name='resnet18', freeze_backbone=True)
model_fe = model_fe.to(DEVICE)

# 统计可训练参数
frozen_params = sum(p.numel() for p in model_fe.parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)

print("=" * 60)
print("特征提取模式 (Feature Extraction)")
print("=" * 60)
print(f"冻结的参数: {frozen_params:,} ({100*frozen_params/(frozen_params+trainable_params):.1f}%)")
print(f"可训练参数: {trainable_params:,} ({100*trainable_params/(frozen_params+trainable_params):.1f}%)")
print(f"\n新的分类头 (fc层):")
print(model_fe.fc)

# 验证冻结状态
print("\n参数 requires_grad 检查:")
print(f"  conv1.weight    requires_grad = {model_fe.conv1.weight.requires_grad}")
print(f"  layer4[1].conv2.weight requires_grad = {model_fe.layer4[1].conv2.weight.requires_grad}")
print(f"  fc[0].weight    requires_grad = {model_fe.fc[0].weight.requires_grad}")
print(f"  fc[3].weight    requires_grad = {model_fe.fc[3].weight.requires_grad}")


## 数据增强（Data Augmentation）

数据增强是防止过拟合的关键技术。通过对训练图片进行随机变换，模型看到的是"不同"的图片，从而学到更鲁棒的特征。

**Albumentations** 是一个强大的图像增强库，基于OpenCV实现，速度快、变换丰富。

In [ ]:
# ===== 数据增强流水线定义 =====

IMG_SIZE = 224

if ALBU_AVAILABLE:
    # ---- 训练数据增强 ----
    train_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.RandomCrop(IMG_SIZE, IMG_SIZE),          # 随机裁剪
        A.HorizontalFlip(p=0.5),                    # 水平翻转
        A.Rotate(limit=20, p=0.5),                  # 随机旋转
        A.ColorJitter(brightness=0.2, contrast=0.2,
                      saturation=0.2, hue=0.1, p=0.5),  # 颜色抖动
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

    # ---- 验证数据增强（不做强增强）----
    val_transform = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
else:
    # Fallback: torchvision transforms
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
    ])
    val_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                            std=[0.229, 0.224, 0.225]),
    ])

print(f"图像尺寸: {IMG_SIZE} x {IMG_SIZE}")
print(f"归一化: ImageNet标准 (mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])")

# ---- 可视化数据增强效果 ----
# 创建演示图片（彩色棋盘格）
demo_img = np.ones((256, 256, 3), dtype=np.uint8) * 200
for i in range(0, 256, 64):
    for j in range(0, 256, 64):
        if (i // 64 + j // 64) % 2 == 0:
            demo_img[i:i+64, j:j+64] = [100, 150, 200]

if ALBU_AVAILABLE:
    vis_transform = A.Compose([
        A.Resize(224, 224),
        A.HorizontalFlip(p=1.0),
        A.Rotate(limit=30, p=1.0),
        A.ColorJitter(brightness=0.3, contrast=0.3, p=1.0),
    ])

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    axes[0].imshow(demo_img)
    axes[0].set_title('原始图片', fontsize=10)
    axes[0].axis('off')

    for i in range(4):
        aug = vis_transform(image=demo_img)['image']
        axes[i+1].imshow(aug)
        axes[i+1].set_title(f'增强样本 {i+1}', fontsize=10)
        axes[i+1].axis('off')

    plt.suptitle('Albumentations 数据增强效果演示', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## 加载数据集

我们将使用Flowers102（牛津花卉数据集）作为演示，它包含102种花卉类别，是迁移学习常用的benchmark数据集。如果下载失败，将回退到CIFAR-10进行演示。

In [ ]:
# ===== 加载数据集 =====

BATCH_SIZE = 16

# 尝试加载Flowers102，失败则使用CIFAR-10
try:
    # Flowers102: 102类花卉，每类40-258张图片
    if ALBU_AVAILABLE:
        # 使用Albumentations时的自定义Dataset包装
        from torchvision.datasets import Flowers102

        class AlbumentationsDataset(torch.utils.data.Dataset):
            def __init__(self, dataset, transform=None):
                self.dataset = dataset
                self.transform = transform

            def __len__(self):
                return len(self.dataset)

            def __getitem__(self, idx):
                image, label = self.dataset[idx]
                image = np.array(image)
                if self.transform:
                    transformed = self.transform(image=image)
                    image = transformed['image']
                return image, label

        raw_train = Flowers102(root='./data', split='train', download=True)
        raw_val = Flowers102(root='./data', split='val', download=True)
        raw_test = Flowers102(root='./data', split='test', download=True)

        train_dataset = AlbumentationsDataset(raw_train, transform=train_transform)
        val_dataset = AlbumentationsDataset(raw_val, transform=val_transform)
        test_dataset = AlbumentationsDataset(raw_test, transform=val_transform)

        num_classes = 102
        class_names = [f'花类{i}' for i in range(102)]
        print(f"Flowers102 数据集加载成功")
except Exception as e:
    print(f"Flowers102 加载失败 ({e})，使用 CIFAR-10 替代")

    # CIFAR-10: 10类，32x32彩色图
    train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
    val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)
    test_dataset = val_dataset

    num_classes = 10
    class_names = ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车']
    print(f"CIFAR-10 数据集加载成功")

# 创建DataLoader
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"训练集: {len(train_dataset)} 张")
print(f"验证集: {len(val_dataset)} 张")
print(f"类别数: {num_classes}")
print(f"批次大小: {BATCH_SIZE}")

# 显示样本
if class_names:
    fig, axes = plt.subplots(2, 5, figsize=(14, 6))
    for i, ax in enumerate(axes.flat):
        img, label = train_dataset[i]
        if isinstance(img, torch.Tensor):
            # 反归一化显示
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            img_disp = img * std + mean
            img_disp = img_disp.permute(1, 2, 0).numpy()
            img_disp = np.clip(img_disp, 0, 1)
        else:
            img_disp = img
        ax.imshow(img_disp)
        ax.set_title(class_names[label][:10] if label < len(class_names) else f'Class {label}', fontsize=9)
        ax.axis('off')
    plt.suptitle('数据集样本', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## 特征提取训练（冻结骨干网络）

在特征提取模式下，只训练新替换的分类头。这适用于数据量较小的情况，训练快速且不易过拟合。

In [ ]:
# ===== 训练函数 =====

def train_one_epoch(model, dataloader, criterion, optimizer, device, phase='train'):
    '''执行一个epoch的训练或验证'''
    if phase == 'train':
        model.train()
    else:
        model.eval()

    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    with torch.set_grad_enabled(phase == 'train'):
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            if phase == 'train':
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects.double().item() / total_samples

    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, num_epochs=3, verbose=True):
    '''完整的模型训练流程'''
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_acc = 0.0
    best_model_state = None

    for epoch in range(num_epochs):
        epoch_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, phase='train'
        )
        val_loss, val_acc = train_one_epoch(
            model, val_loader, criterion, optimizer, device, phase='val'
        )

        if isinstance(scheduler, lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            best_model_state = copy.deepcopy(model.state_dict())

        if verbose:
            print(f"Epoch {epoch+1:2d}/{num_epochs} | "
                  f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
                  f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
                  f"LR: {current_lr:.2e} | {(time.time()-epoch_start):.1f}s")

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        if verbose:
            print(f"\n加载最佳模型 (最佳验证准确率: {best_acc:.4f})")

    return model, history


# ===== 特征提取模式训练 =====
print("=" * 60)
print("阶段1：特征提取训练 (冻结骨干网络)")
print("=" * 60)

model_fe = build_transfer_model(num_classes, model_name='resnet18', freeze_backbone=True)
model_fe = model_fe.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer_fe = optim.Adam(
    filter(lambda p: p.requires_grad, model_fe.parameters()),
    lr=1e-3
)
scheduler_fe = lr_scheduler.ReduceLROnPlateau(
    optimizer_fe, mode='min', factor=0.5, patience=1, verbose=False
)

model_fe, history_fe = train_model(
    model_fe, train_loader, val_loader,
    criterion, optimizer_fe, scheduler_fe,
    DEVICE, num_epochs=3
)

print(f"\n特征提取阶段完成!")
print(f"  最佳验证准确率: {max(history_fe['val_acc']):.4f}")


## 微调（Fine-Tuning）

在特征提取训练获得良好性能后，我们可以解冻骨干网络的部分层（通常是最后几层），用更小的学习率继续训练。这样模型可以适应目标数据集的细微特征差异。

In [ ]:
# ===== 微调阶段 =====

print("=" * 60)
print("阶段2：微调 (解冻layer3和layer4)")
print("=" * 60)

# 解冻最后两个layer
for name, param in model_fe.named_parameters():
    if 'layer3' in name or 'layer4' in name:
        param.requires_grad = True
    if 'fc' in name:
        param.requires_grad = True  # 保持分类头可训练

# 统计可训练参数
ft_trainable = sum(p.numel() for p in model_fe.parameters() if p.requires_grad)
ft_total = sum(p.numel() for p in model_fe.parameters())
print(f"可训练参数: {ft_trainable:,} / {ft_total:,} ({100*ft_trainable/ft_total:.1f}%)")

# 分层学习率：分类头用较大学习率，微调的卷积层用较小学习率
optimizer_ft = optim.Adam([
    {'params': [p for n, p in model_fe.named_parameters() if 'fc' in n and p.requires_grad],
     'lr': 1e-3},
    {'params': [p for n, p in model_fe.named_parameters() if 'fc' not in n and p.requires_grad],
     'lr': 1e-4},
])

scheduler_ft = lr_scheduler.ReduceLROnPlateau(
    optimizer_ft, mode='min', factor=0.5, patience=1, verbose=False
)

model_fe, history_ft = train_model(
    model_fe, train_loader, val_loader,
    criterion, optimizer_ft, scheduler_ft,
    DEVICE, num_epochs=2
)

# 对比特征提取和微调的效果
print(f"\n效果对比:")
print(f"  特征提取最佳验证准确率: {max(history_fe['val_acc']):.4f}")
print(f"  微调后最佳验证准确率:   {max(history_ft['val_acc']):.4f}")

# 绘制训练曲线
combined_train_loss = history_fe['train_loss'] + history_ft['train_loss']
combined_val_loss = history_fe['val_loss'] + history_ft['val_loss']
combined_train_acc = history_fe['train_acc'] + history_ft['train_acc']
combined_val_acc = history_fe['val_acc'] + history_ft['val_acc']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(combined_train_loss) + 1)

axes[0].plot(epochs, combined_train_loss, 'b-o', label='训练损失', markersize=4)
axes[0].plot(epochs, combined_val_loss, 'r-s', label='验证损失', markersize=4)
axes[0].axvline(x=len(history_fe['train_loss']) + 0.5, color='gray', linestyle='--', alpha=0.7)
axes[0].text(len(history_fe['train_loss']) + 0.5, axes[0].get_ylim()[1] * 0.9,
             '← 特征提取 | 微调 →', ha='center', fontsize=9, color='gray')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('损失曲线')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, combined_train_acc, 'b-o', label='训练准确率', markersize=4)
axes[1].plot(epochs, combined_val_acc, 'r-s', label='验证准确率', markersize=4)
axes[1].axvline(x=len(history_fe['train_acc']) + 0.5, color='gray', linestyle='--', alpha=0.7)
axes[1].text(len(history_fe['train_acc']) + 0.5, axes[1].get_ylim()[1] * 0.9,
             '← 特征提取 | 微调 →', ha='center', fontsize=9, color='gray')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('准确率曲线')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('迁移学习 — 特征提取 + 微调 完整训练过程', fontsize=14)
plt.tight_layout()
plt.show()


## Grad-CAM可视化

**Grad-CAM（Gradient-weighted Class Activation Mapping）** 可以显示CNN在做分类决策时"关注"图片的哪些区域。通过计算目标类别得分对最后一个卷积层特征图的梯度，我们可以生成热力图来可视化模型的注意力分布。

In [ ]:
# ===== Grad-CAM 实现 =====

class GradCAM:
    '''Grad-CAM 可视化 — 显示模型的关注区域'''

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.hook_handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.hook_handles.append(
            self.target_layer.register_forward_hook(forward_hook)
        )
        self.hook_handles.append(
            self.target_layer.register_full_backward_hook(backward_hook)
        )

    def __call__(self, input_tensor, target_class=None):
        self.model.eval()
        self.model.zero_grad()

        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1).squeeze(0)
        cam = torch.relu(cam)

        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.cpu().numpy()

    def remove_hooks(self):
        for handle in self.hook_handles:
            handle.remove()
        self.hook_handles.clear()


def apply_heatmap(image, heatmap, alpha=0.5, colormap='jet'):
    '''将热力图叠加到原图上'''
    from PIL import Image as PILImage
    heatmap_resized = np.array(PILImage.fromarray(
        (heatmap * 255).astype(np.uint8)
    ).resize((image.shape[1], image.shape[0]), PILImage.BILINEAR)) / 255.0

    cmap = plt.get_cmap(colormap)
    heatmap_colored = cmap(heatmap_resized)[:, :, :3]
    heatmap_colored = (heatmap_colored * 255).astype(np.uint8)

    overlay = (image.astype(float) * (1 - alpha) + heatmap_colored.astype(float) * alpha)
    overlay = np.clip(overlay, 0, 255).astype(np.uint8)

    return overlay


print("Grad-CAM类和热力图叠加函数定义完成")

# ---- Grad-CAM演示 ----
model_fe.eval()
target_layer = model_fe.layer4[1].conv2  # ResNet18最后一个卷积层

# 取几张验证集图片做Grad-CAM
fig, axes = plt.subplots(2, 4, figsize=(16, 9))

for i in range(4):
    img, label = val_dataset[i]
    input_tensor = img.unsqueeze(0).to(DEVICE)

    # 获取预测
    with torch.no_grad():
        output = model_fe(input_tensor)
        pred = output.argmax(dim=1).item()

    # Grad-CAM
    grad_cam = GradCAM(model_fe, target_layer)
    heatmap = grad_cam(input_tensor, target_class=label)

    # 准备显示原图
    if isinstance(img, torch.Tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_disp = img.cpu() * std + mean
        img_disp = (img_disp.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    else:
        img_disp = img

    overlay = apply_heatmap(img_disp, heatmap, alpha=0.5)

    true_name = class_names[label] if label < len(class_names) else f'类{label}'
    pred_name = class_names[pred] if pred < len(class_names) else f'类{pred}'

    axes[0, i].imshow(img_disp)
    axes[0, i].set_title(f'真实: {true_name}', fontsize=9)
    axes[0, i].axis('off')

    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f'Grad-CAM\n预测: {pred_name}', fontsize=9)
    axes[1, i].axis('off')

    grad_cam.remove_hooks()

axes[0, 0].set_ylabel('原图', fontsize=12)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=12)
plt.suptitle('Grad-CAM 热力图 — 模型关注区域可视化', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Grad-CAM 解读:")
print("  红色/黄色 = 模型重点关注的区域（对分类决策贡献大）")
print("  蓝色 = 模型不太关注的区域")


In [ ]:
# ===== 分类报告与混淆矩阵 =====

from sklearn.metrics import classification_report, confusion_matrix

model_fe.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(DEVICE)
        outputs = model_fe(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

# 分类报告
print("分类报告:")
target_names = [class_names[i] for i in sorted(set(all_labels))]
print(classification_report(all_labels, all_preds, target_names=target_names, digits=3))

# 混淆矩阵
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.colorbar(im, ax=ax, shrink=0.8)

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        if cm[i, j] > 0:
            ax.text(j, i, str(cm[i, j]),
                    ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=7)

ax.set_xlabel('预测类别', fontsize=12)
ax.set_ylabel('真实类别', fontsize=12)
ax.set_title('混淆矩阵', fontsize=14)
ax.set_xticks(range(len(target_names)))
ax.set_yticks(range(len(target_names)))
ax.set_xticklabels(target_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(target_names, fontsize=8)
plt.tight_layout()
plt.show()


## 练习与实践

In [ ]:
# TODO: 练习1 - 尝试不同的预训练模型
# 比较EfficientNet、DenseNet等模型的迁移学习效果

print("练习1: 尝试不同的预训练模型")
print("=" * 50)

# 可用的模型列表
available_models = ['resnet18', 'resnet50', 'resnet152',
                    'densenet121', 'efficientnet_b0', 'mobilenet_v3_large']

print("torchvision 提供的预训练模型:")
for m in available_models:
    print(f"  - {m}")

print("\nTODO: 选择一个模型，替换到 build_transfer_model 中，对比训练效果")
print("\n需要关注的指标:")
print("  1. 模型参数量（越大越慢）")
print("  2. 训练时间和显存占用")
print("  3. 验证准确率")
print("  4. 推理速度（FPS）")

# 示例：快速对比不同模型的参数量
print(f"\n模型参数量对比 (ImageNet预训练):")
for model_name in ['resnet18', 'resnet50', 'densenet121']:
    try:
        if model_name == 'resnet18':
            m = models.resnet18()
        elif model_name == 'resnet50':
            m = models.resnet50()
        elif model_name == 'densenet121':
            m = models.densenet121()
        params = sum(p.numel() for p in m.parameters())
        print(f"  {model_name:<20} {params:>12,} 参数")
    except:
        print(f"  {model_name:<20} 加载失败")

print("\n提示: 更大的模型不一定更好，需要根据数据和任务选择。")


In [ ]:
# TODO: 练习2 - 修改数据增强策略
# 观察不同数据增强对模型泛化能力的影响

print("练习2: 修改数据增强策略")
print("=" * 50)

# 定义不同的增强策略
aug_strategies = {
    '基础增强': ['RandomHorizontalFlip', 'RandomRotation'],
    '中等增强': ['RandomHorizontalFlip', 'RandomRotation', 'ColorJitter'],
    '强增强': ['RandomHorizontalFlip', 'RandomRotation', 'ColorJitter',
             'RandomErasing', 'RandomAffine'],
    'Cutout': ['RandomHorizontalFlip', 'CoarseDropout (随机遮挡)'],
}

print("不同数据增强策略:")
for name, augs in aug_strategies.items():
    print(f"  {name}: {', '.join(augs)}")

print("\nTODO:")
print("  1. 选择3种不同的增强策略分别训练模型")
print("  2. 对比验证准确率")
print("  3. 分析哪种策略最适合当前数据集")
print("\n关键问题:")
print("  - 强增强是否总是更好？")
print("  - 增强强度与数据集大小有什么关系？")
print("  - 过度增强有什么负面影响？")


## 实战案例：花卉种类识别应用

结合迁移学习的所有技术，构建一个花卉识别系统。演示完整的推理流程：加载模型 → 图片预处理 → 预测 → 显示Top-3结果 → Grad-CAM可解释性分析。

In [ ]:
# ===== 实战：花卉种类识别应用 =====

model_fe.eval()
model_fe = model_fe.to(DEVICE)

print("=" * 60)
print("花卉识别应用演示")
print("=" * 60)

# 从测试集中取几张图片进行预测
num_samples = 4
fig, axes = plt.subplots(2, num_samples, figsize=(5*num_samples, 10))

for i in range(num_samples):
    idx = i * (len(test_dataset) // num_samples)
    img, true_label = test_dataset[idx]
    input_tensor = img.unsqueeze(0).to(DEVICE)

    # 推理
    with torch.no_grad():
        output = model_fe(input_tensor)
        probs = torch.softmax(output, dim=1)
        top3_probs, top3_indices = torch.topk(probs, 3, dim=1)

    true_name = class_names[true_label] if true_label < len(class_names) else f'类{true_label}'

    # 准备显示图片
    if isinstance(img, torch.Tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        img_disp = img.cpu() * std + mean
        img_disp = (img_disp.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    else:
        img_disp = img

    # 原图 + Top-3预测
    axes[0, i].imshow(img_disp)
    axes[0, i].set_title(f'真实: {true_name}', fontsize=10)
    axes[0, i].axis('off')

    # Grad-CAM
    target_layer = model_fe.layer4[1].conv2
    grad_cam = GradCAM(model_fe, target_layer)
    heatmap = grad_cam(input_tensor, target_class=true_label)
    overlay = apply_heatmap(img_disp, heatmap, alpha=0.5)

    axes[1, i].imshow(overlay)
    axes[1, i].set_title('Grad-CAM', fontsize=10)
    axes[1, i].axis('off')

    grad_cam.remove_hooks()

    # 打印Top-3
    top3_names = [class_names[idx] if idx < len(class_names) else f'类{idx}'
                  for idx in top3_indices[0].cpu().numpy()]
    top3_p = top3_probs[0].cpu().numpy()
    print(f"\n样本 {i+1} (真实: {true_name}):")
    for rank, (name, prob) in enumerate(zip(top3_names, top3_p)):
        marker = ' [正确!]' if name == true_name else ''
        print(f"  Top-{rank+1}: {name} ({prob:.2%}){marker}")

axes[0, 0].set_ylabel('原图', fontsize=12)
axes[1, 0].set_ylabel('Grad-CAM', fontsize=12)
plt.suptitle('花卉识别应用 — 预测结果与可解释性', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("\n应用总结:")
print("  1. 迁移学习使得小数据集上也能训练出高精度分类器")
print("  2. 特征提取 + 微调的组合策略通常效果最好")
print("  3. Grad-CAM提供了模型决策的可解释性")
print("  4. 这种技术可直接应用于医疗影像、工业质检、自动驾驶等领域")
